In [11]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Feature Extraction Function
def extract_features(df, window_size=80, step_size=40):
    features = []
    labels = []
    for i in range(0, len(df) - window_size, step_size):
        window = df[['x-acc', 'y-acc', 'z-acc']].iloc[i:i+window_size].values
        means = np.mean(window, axis=0)
        stds = np.std(window, axis=0)
        maxs = np.max(window, axis=0)
        mins = np.min(window, axis=0)
        sma = np.mean(np.sum(np.abs(window), axis=1))
        
        feat_vector = np.concatenate([means, stds, maxs, mins, [sma]])
        features.append(feat_vector)
        labels.append(df['Activity'].iloc[i])
    return np.array(features), np.array(labels)

# 2. Load and Process Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

X_train_raw, y_train_strings = extract_features(train_df)
X_test_raw, y_test_strings = extract_features(test_df)

le = LabelEncoder()
y_train = le.fit_transform(y_train_strings)
y_test = le.transform(y_test_strings)

print("Label Mapping:", dict(zip(le.classes_, range(len(le.classes_)))))

# 3. Scaling (CRITICAL)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# --- COPY THESE INTO YOUR MCU C++ CODE ---
print("\n--- COPY TO MCU ---")
print("float scaler_mean[13] = {" + ", ".join(map(str, scaler.mean_)) + "};")
print("float scaler_std[13] = {" + ", ".join(map(str, np.sqrt(scaler.var_))) + "};")
print("-------------------\n")

# 4. Build and Train Model
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(13,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(len(le.classes_), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test), verbose=1)

# 5. Quantization with Shuffled Representative Dataset
def representative_data_gen():
    # We shuffle to ensure the model sees all activities during calibration
    idx = np.arange(len(X_train))
    np.random.shuffle(idx)
    for i in idx[:200]:
        sample = np.expand_dims(X_train[i], axis=0).astype(np.float32)
        yield [sample]

# --- Python Conversion Code ---
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# REMOVE the optimizations and representative dataset lines
# This forces the model to stay in Float32 mode
tflite_model = converter.convert()

with open('har_model.tflite', 'wb') as f:
    f.write(tflite_model)

# Verify accuracy (should be ~92%)
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
in_idx = interpreter.get_input_details()[0]['index']
out_idx = interpreter.get_output_details()[0]['index']

hits = 0
for i in range(len(X_test)):
    interpreter.set_tensor(in_idx, X_test[i:i+1].astype(np.float32))
    interpreter.invoke()
    if np.argmax(interpreter.get_tensor(out_idx)) == y_test[i]:
        hits += 1
print(f"✅ Float32 TFLite Accuracy: {hits/len(y_test)*100:.2f}%")


# 7. Generate Header
with open('har_model_data.h', 'w') as f:
    f.write('unsigned char har_model_data[] = {\n')
    f.write(', '.join([f'0x{b:02x}' for b in tflite_model]))
    f.write(f'\n}};\nunsigned int har_model_data_len = {len(tflite_model)};')

Label Mapping: {np.int64(0): 0, np.int64(1): 1, np.int64(2): 2, np.int64(3): 3, np.int64(4): 4, np.int64(5): 5}

--- COPY TO MCU ---
float scaler_mean[13] = {0.51492155958734, 0.6800644083957155, 0.5139762513951088, 0.07422821042691657, 0.08259200361197701, 0.06597407728290483, 0.6646638849324351, 0.8338006332349814, 0.6836338181333433, 0.3699998136573051, 0.5101317889780111, 0.38362613418834807, 1.7089622193781666};
float scaler_std[13] = {0.11348377672350908, 0.09627593933077606, 0.06950648134687744, 0.04647531713452688, 0.042726408946011, 0.03095496039363151, 0.14083906671786833, 0.10192102216221496, 0.0804269672720422, 0.13975024851788742, 0.1377396651492351, 0.11796965116130931, 0.1562311717581697};
-------------------

Epoch 1/50
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 1s 823us/step - accuracy: 0.7852 - loss: 0.6479 - val_accuracy: 0.8418 - val_loss: 0.4384
Epoch 2/50
1275/1275 ━━━━━━━━━━━━━━━━━━━━ 1s 765us/step - accuracy: 0.8609 - loss: 0.4011 - val_accuracy: 0.8702 - val_loss: 0.3748
E

INFO:tensorflow:Assets written to: C:\Users\Eyll\AppData\Local\Temp\tmpo0votjhy\assets


Saved artifact at 'C:\Users\Eyll\AppData\Local\Temp\tmpo0votjhy'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 13), dtype=tf.float32, name='keras_tensor_8')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  1941930469392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1941930474768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1942000545296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1942000545488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1942000536464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1942000545680: TensorSpec(shape=(), dtype=tf.resource, name=None)
✅ Float32 TFLite Accuracy: 92.03%


c:\Users\Eyll\Codes\PythonCodes\CSE421HW3\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
